## Import CSV's and Python Libraries

Run `pip install -r requirements.txt` in your terminal to install the packages necessary to run this notebook on your local machine

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv("csv's/mlb_2024-2026.csv")

## Clean Data

In [3]:
# Map StatCast variables to Trackman variables

Stat_Map = {
    #'release_speed' : 'release_speed',
    'release_spin_rate' : 'spin_rate',
    'pfx_z' : 'induced_vertical_break',
    'pfx_x' : 'horizontal_break',
    'release_extension' : 'extension',
    'release_pos_z' : 'release_height',
    'release_pos_x' : 'release_side',
    'release_spin_axis' : 'spin_axis',          
}

# Flip LHP metrics so that all pitchers are treated the same (as RHP)
Flip_Cols = [
    'horizontal_break',
    'release_side',
]

FEATURES = ['release_speed', 'spin_rate', 'induced_vertical_break',
            'horizontal_break', 'extension', 'release_height',
            'release_side', 'spin_axis']

df = df.rename(columns=Stat_Map)

# Flip the LHP metrics so that all pitches are treated as RHP
df.loc[df['p_throws'] == 'L', Flip_Cols] *= -1
#Flip spin axis
df.loc[df['p_throws'] == 'L', 'spin_axis'] = (360 - df.loc[df['p_throws'] == 'L', 'spin_axis']) % 360

# Drop rows where target variable is null
df = df.dropna(subset=['delta_run_exp'])

#Split data into two groups
train = df[df['game_date'] < '2026-01-01'].copy()
test = df[df['game_date'] >= '2026-01-01'].copy()

x_train = train[FEATURES]
y_train = train['delta_run_exp']
x_test  = test[FEATURES]
y_test  = test['delta_run_exp']


## Training MLBstuff+ model

Below is the cell that actually runs the regression model to generate "MLBstuff+". This model is trained on real MLB StatCast data from 2024 up through the present day (May 2026).

It will use a XGBoost regression model using Run Values as the target variable. Specifically, the change (or delta) in run value per pitch will be calculated.

As the model is being trained, the features (inputs) will be weighted accordingly to produce the most accurate stuff+ model.

### Primary Goal

MLBstuff+ will be measured just like any other stuff+ model with 100 being MLB average. The scoring has been standardized in a way such that 10 points above or below 100 is one standard deviation. For example if a pitcher has a slider that scores at 110, that means his slider is 10% better than MLB average or 1 standard deviation above average.

### Target Variable

RV = Run Value
xRV = Expected Run Value
ΔRV = The change in Run Value
Δ = delta

ΔRV is the target variable that was used to train the regression model for MLBstuff+. Rows in the StatCast CSV where ΔRV is not present were dropped. It is calculated using the following formula:

ΔRV = RV<sub>new</sub> - RV<sub>old</sub>

This is done for the entire StatCast dataset that the model was trained on, and it calculates the appropriate weights for the features to determine what matters when it comes to getting hitters out. Features that matter a lot are weighted heavily in the model whereas features that do not are weighted less.

### Features/Inputs

The features that were pulled in were chosen because they have something to do with the raw "stuff" the pitcher has. Anything that might have an impact on a pitcher's stuff was included. The full list of features is as follows

**Features:**
- `release_speed`
- `spin_rate`
- `induced_vertical_break`
- `horizontal_break`
- `extension`
- `release_height`
- `release_side`
- `spin_axis`

### Important Notes

Unfortunately, spin efficiency is not included in StatCast and can't be included in the model for that reason. Also, VAA and HAA are not included in StatCast either as these are calculated using physics models. That is something that can potentially be incorporated in the future.

In [4]:
#Generated code for the time being

model = XGBRegressor(
    n_estimators=500,      # number of trees — more = better fit but slower
    learning_rate=0.05,    # how much each tree contributes — lower = more conservative, needs more trees
    max_depth=5,           # how deep each tree can be — controls complexity
    subsample=0.8,         # fraction of rows sampled per tree — helps prevent overfitting
    random_state=42
)

model.fit(
    x_train, y_train,
    eval_set=[(x_test, y_test)],  # prints validation loss as it trains
    verbose=100                    # print every 100 trees
)

# Check accuracy on test set
y_pred = model.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

[0]	validation_0-rmse:0.22131
[100]	validation_0-rmse:0.22124
[200]	validation_0-rmse:0.22124
[300]	validation_0-rmse:0.22125
[400]	validation_0-rmse:0.22126
[499]	validation_0-rmse:0.22128
Test RMSE: 0.2213


## Standardizing the Data

In [5]:
#Generated for the time being

# Predict a raw ΔRV score for every pitch
df['predicted_rv'] = model.predict(df[FEATURES])

# Aggregate: mean predicted ΔRV per pitcher per pitch type
scores = df.groupby(['player_name', 'pitch_type'])['predicted_rv'].mean().reset_index()

# Normalize to 100 mean, 10 = 1 std dev
# Lower (more negative) predicted_rv = better pitch = higher stuff+, so we flip the sign
pop_mean = scores['predicted_rv'].mean()
pop_std  = scores['predicted_rv'].std()

scores['stuff_plus'] = 100 + ((pop_mean - scores['predicted_rv']) / pop_std) * 10

scores.sort_values('stuff_plus', ascending=False).head(20)

C:\Users\Conner\AppData\Local\Temp\ipykernel_27164\1838542186.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['predicted_rv'] = model.predict(df[FEATURES])


,player_name,pitch_type,predicted_rv,stuff_plus
6378,"Snell, Blake",SI,-0.082689,169.662201
7090,"Vázquez, Luis",SL,-0.068568,158.207428
1891,"Estrada, Jeremiah",SI,-0.037419,132.939148
4300,"McGuire, Reese",SL,-0.028678,125.848526
5409,"Pushard, Matt",SI,-0.022470,120.812866
6999,"Varland, Gus",SI,-0.020409,119.141098
5789,"Rogers, Tyler",SI,-0.019050,118.038567
4329,"McMillon, John",FC,-0.018441,117.544083
2664,"Halvorsen, Seth",PO,-0.018350,117.470383
4125,"Mastrobuoni, Miles",FA,-0.017378,116.681847


In [6]:
df[FEATURES + ['delta_run_exp']].isnull().sum()

release_speed             30452
spin_rate                 36858
induced_vertical_break    30453
horizontal_break          30586
extension                 32328
release_height            30451
release_side              30451
spin_axis                 36860
delta_run_exp                 0
dtype: int64